# 포트폴리오 RAG 테스트 (노트북)

노트북에서 RAG 연결·API 키·인덱스·답변을 단계별로 확인합니다.  
**사전 준비**: 프로젝트 루트에 `.env` (OPENAI_API_KEY), `data/portfolio/` 에 PDF/Word 넣은 뒤 `uv run python scripts/build_index.py` 로 인덱스 생성.

In [1]:
print('Hello from test_rag.ipynb!')

Hello from test_rag.ipynb!


In [2]:
# 1) 경로 추가 + .env 로드
import sys
from pathlib import Path

# 노트북 실행 위치가 notebooks/ 이면 상위가 프로젝트 루트
ROOT = Path().resolve()
if (ROOT / ".env").exists():
    pass
elif (ROOT.parent / ".env").exists():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError(".env 를 찾을 수 없습니다. 프로젝트 루트(scy_Rag)에서 노트북을 열거나, 루트를 ROOT에 넣어주세요.")
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import os
key = os.getenv("OPENAI_API_KEY")
print(f"OPENAI_API_KEY 설정됨: {bool(key and key.strip())}")
print(f"프로젝트 루트: {ROOT}")

OPENAI_API_KEY 설정됨: True
프로젝트 루트: C:\Users\User\Project\agent_portfolio\scy_Rag


In [3]:
# 2) 인덱스 존재 여부 확인
index_dir = ROOT / "index"
faiss_path = index_dir / "index.faiss"
exists = faiss_path.exists()
print(f"인덱스 존재: {exists}")
if not exists:
    print("→ 터미널에서: uv run python scripts/build_index.py")

인덱스 존재: True


In [4]:
# 3) RAG 한 번 호출 (get_answer)
from app.rag import get_answer

question = "이 사람의 강점을 한 줄로 요약해 주세요."
print(f"질문: {question}")
print("-" * 50)

answer, source_docs = get_answer(question)
print(answer)
print("-" * 50)
print(f"참고 문단 수: {len(source_docs)}")

c:\Users\User\Project\agent_portfolio\scy_Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


질문: 이 사람의 강점을 한 줄로 요약해 주세요.
--------------------------------------------------


C:\Users\User\Project\agent_portfolio\scy_Rag\app\rag.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 873.02it/s, Materializing param=pooler.dense.weight]                               
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


지원자는 빅데이터와 머신러닝에 대한 깊은 관심을 바탕으로, 광고 소재 제작에 있어 생성형 AI 모델을 활용하여 효과적인 결과를 도출하는 능력을 갖추고 있습니다.
--------------------------------------------------
참고 문단 수: 4


In [5]:
# 4) 스트리밍으로 답변 받기 (get_answer_stream)
from app.rag import get_answer_stream

question = "주요 경력과 프로젝트를 간단히 소개해 주세요."
print(f"질문: {question}")
print("-" * 50)

for full, source_docs in get_answer_stream(question):
    pass  # 마지막 full만 사용
print(full)
print("-" * 50)
print(f"참고 문단 수: {len(source_docs)}")

질문: 주요 경력과 프로젝트를 간단히 소개해 주세요.
--------------------------------------------------
주요 경력으로는 2023년 1월부터 4월까지 에코마케팅에서 인턴으로 근무하였고, 2024년 1월부터 현재까지 이엠넷 R&D 본부에서 활동하고 있습니다. 

프로젝트로는 이엠넷 사내 솔루션인 "ChatGPT를 이용한 광고 소재 제작 솔루션"을 개발하였으며, 이 프로젝트에서 ChatGPT의 파인 튜닝과 Midjourney 결과물 데이터화 및 분석을 담당했습니다. 이 솔루션을 통해 AI로 생성한 광고 소재의 CTR을 약 10% 개선하고 광고주 재계약에 기여하였습니다. 

또한, 모두의 연구소에서 AI/LLM 서비스 개발 과정에 참여하고 있으며, QMS 데이터 분석 학회에서도 활동하고 있습니다.
--------------------------------------------------
참고 문단 수: 4


In [8]:
# 5) 검색만 테스트 (retriever만 사용)
import config
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
)
vectorstore = FAISS.load_local(
    str(config.INDEX_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

docs = retriever.invoke("AI")
for i, d in enumerate(docs, 1):
    print(f"[문단 {i}] {d.page_content[:200]}...")
    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1292.61it/s, Materializing param=pooler.dense.weight]                               
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[문단 1] ﻿# AI 프로젝트
## 사내 솔루션 Q&A 챗봇 설계
Preview
* 이엠넷 사내 솔루션
* 목적 : 사내 Power Automate와 Copilot Studio, AWS SageMaker를 활용하여, 비용 문의·과금 체계 안내 및 광고 성과 예측을 자동 처리하는 AI 챗봇(에이전트) 구축
* 역할 : Copilot Studio 환경 설계 및 ...

[문단 2] * 각 브랜드 특성에 맞는 DA 생성하는 생성형 AI 모델 제작
            * AI로 생성한 DA 소재 CTR 10% 가량 개선
            * 광고주 재계약에 기여
5. 한계점
            * 간간단한 명령어로 광고 이미지를 제작할 수 있다는 점이 경쟁력이었지만, ChatGPT의 이미지 생성 기능이 고도화되면서 마케터들...

